# 02 — Retrieval Component Evaluation

Dense backbone (Table 16), lexical/semantic/hybrid (Table 17), Kazakh character
folding (Table 18), and second-stage reranking ablation (Table 19).

Assumes the corpus/benchmark objects from `01_corpus_construction.ipynb` are available
(run 01 first in the same session, or factor shared loaders into src/).


In [ ]:
# Configuration is centralised in src/config.py (single source of truth).
# If running outside the package, the constants below mirror that file.
import sys, os
sys.path.append(os.path.abspath(".."))
try:
    from src.config import *
    print("Loaded config from src.config")
except Exception as e:
    print("src.config not importable here; using inline constants.", e)
    RANDOM_SEED = 42
    RRF_KAPPA = 60
    DENSE_BATCH_SIZE = 128
    MAX_SEQ_LENGTH = 512
    NAIVE_K, STATIC_K = 5, 10
    ADAPTIVE_K = {"simple":5,"moderate":10,"complex":15}


## Retrieval helpers: metrics, BM25, dense, RRF (paper Eq. 4, kappa=60)

In [ ]:
#@title 3.1 Retrieval result object and metrics

def make_result(indices, scores=None, score_name='score'):
    ids, docs, srcs, texts, titles, scs = [], [], [], [], [], []
    if scores is None:
        scores = [np.nan] * len(indices)
    for idx, score in zip(indices, scores):
        idx = int(idx)
        if idx < 0 or idx >= len(c_ids):
            continue
        ids.append(c_ids[idx])
        docs.append(c_docs[idx])
        srcs.append(c_srcs[idx])
        texts.append(c_text[idx])
        titles.append(c_titles[idx])
        scs.append(float(score) if score is not None and not pd.isna(score) else np.nan)
    return {
        'retrieved_ids': ids,
        'retrieved_docs': docs,
        'retrieved_sources': srcs,
        'retrieved_texts': texts,
        'retrieved_titles': titles,
        'retrieved_scores': scs,
        'score_name': score_name,
    }


def hit_at_1(ids, gold_set):
    return int(bool(ids) and ids[0] in gold_set)


def recall_at_k(ids, gold_set, k):
    if not gold_set:
        return np.nan
    return len(set(ids[:k]) & gold_set) / len(gold_set)


def mrr_at_k(ids, gold_set, k=10):
    for rank, cid in enumerate(ids[:k], 1):
        if cid in gold_set:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ids, gold_set, k=10):
    if not gold_set:
        return np.nan
    dcg = sum(1.0 / np.log2(rank + 1) for rank, cid in enumerate(ids[:k], 1) if cid in gold_set)
    ideal_hits = min(len(gold_set), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg else 0.0


def exact_value_recall_at_k(values, gold_value, k):
    gold_value = clean_str(gold_value)
    if not gold_value:
        return np.nan
    return int(gold_value in [clean_str(v) for v in values[:k]])


def eval_retrieval(results, bench, kset=(1, 3, 5, 10)):
    """Evaluate retrieval on answerable questions with gold_chunk_id/gold_doc_id/gold_source."""
    rows = []
    for res, (_, row) in zip(results, bench.iterrows()):
        # FIX: support one OR several gold chunk ids separated by | ; ,
        gold_chunk = {p for p in re.split(r'[|;,]\s*', clean_str(row.get('gold_chunk_id_str', ''))) if p} - {''}
        ids = [clean_str(x) for x in res.get('retrieved_ids', [])]
        docs = [clean_str(x) for x in res.get('retrieved_docs', [])]
        srcs = [clean_str(x) for x in res.get('retrieved_sources', [])]
        scores = res.get('retrieved_scores', [])
        out = {
            'question_id': row.get('question_id', ''),
            'row_id': row.get('row_id', ''),
            'complexity': row.get('complexity', ''),
            'question_type': row.get('question_type', ''),
            'legal_domain': row.get('legal_domain', ''),
            'Hit@1': hit_at_1(ids, gold_chunk),
            'MRR@10': mrr_at_k(ids, gold_chunk, 10),
            'nDCG@10': ndcg_at_k(ids, gold_chunk, 10),
            'DocRecall@10': exact_value_recall_at_k(docs, row.get('gold_doc_id_str', ''), 10),
            'SourceRecall@10': exact_value_recall_at_k(srcs, row.get('gold_source_str', ''), 10),
            'TopScore': float(scores[0]) if scores else np.nan,
            'TopChunk': ids[0] if ids else '',
            'GoldChunk': next(iter(gold_chunk)) if gold_chunk else '',
        }
        for k in kset:
            out[f'Recall@{k}'] = recall_at_k(ids, gold_chunk, k)
        rows.append(out)
    details = pd.DataFrame(rows)
    metric_cols = [
        'Hit@1', 'Recall@1', 'Recall@3', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10',
        'DocRecall@10', 'SourceRecall@10', 'TopScore'
    ]
    agg = {c: round(float(np.nanmean(details[c])), 4) for c in metric_cols if c in details.columns}
    return agg, details


def summarize_metric_table(rows, name, sort_col='Recall@10'):
    df = pd.DataFrame(rows)
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    display(df)
    save_table(df, name)
    return df

print('Metrics ready.')

In [ ]:
#@title 3.2 Tokenization, BM25, dense retrieval, and RRF helpers
from rank_bm25 import BM25Okapi


def tok(text):
    return str(text).lower().split()


def build_bm25(texts, tokenizer=tok, desc='BM25 corpus'):
    tokenized = [tokenizer(t) for t in tqdm(texts, desc=desc)]
    return BM25Okapi(tokenized)


def bm25_ret(index, query, k=10, tokenizer=tok):
    scores = index.get_scores(tokenizer(query))
    top = np.argsort(scores)[::-1][:k]
    return make_result(top, scores[top], score_name='bm25')


def dense_ret(index, query_embedding, k=10):
    scores, idx = index.search(query_embedding.reshape(1, -1).astype('float32'), k)
    return make_result(idx[0], scores[0], score_name='dense_ip')


def rrf(result_list, k=10, rrf_k=RRF_K):
    scores = defaultdict(float)
    meta = {}
    for res in result_list:
        ids = res.get('retrieved_ids', [])
        for rank, cid in enumerate(ids, 1):
            scores[cid] += 1.0 / (rrf_k + rank)
            pos = ids.index(cid)
            meta[cid] = {
                'doc': res.get('retrieved_docs', [''])[pos] if pos < len(res.get('retrieved_docs', [])) else '',
                'source': res.get('retrieved_sources', [''])[pos] if pos < len(res.get('retrieved_sources', [])) else '',
                'text': res.get('retrieved_texts', [''])[pos] if pos < len(res.get('retrieved_texts', [])) else '',
                'title': res.get('retrieved_titles', [''])[pos] if pos < len(res.get('retrieved_titles', [])) else '',
            }
    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return {
        'retrieved_ids': top_ids,
        'retrieved_docs': [meta[cid]['doc'] for cid in top_ids],
        'retrieved_sources': [meta[cid]['source'] for cid in top_ids],
        'retrieved_texts': [meta[cid]['text'] for cid in top_ids],
        'retrieved_titles': [meta[cid]['title'] for cid in top_ids],
        'retrieved_scores': [scores[cid] for cid in top_ids],
        'score_name': 'rrf',
    }


def retrieval_confidence(res):
    scores = res.get('retrieved_scores', [])
    return float(scores[0]) if scores else 0.0

print('Retrieval utilities ready.')

In [ ]:
#@title 3.2-fix rrf with correct text/doc alignment (no ids.index lookup)
def rrf(result_list, k=10, rrf_k=RRF_K):
    scores = defaultdict(float)
    meta = {}
    for res in result_list:
        ids    = res.get('retrieved_ids', [])
        docs   = res.get('retrieved_docs', [])
        srcs   = res.get('retrieved_sources', [])
        texts  = res.get('retrieved_texts', [])
        titles = res.get('retrieved_titles', [])
        for rank, cid in enumerate(ids):          # rank position = list index, synchronous
            scores[cid] += 1.0 / (rrf_k + rank + 1)
            # FIX: take metadata at THIS position (rank), not via ids.index(cid).
            # Only set if not already present, so the first retriever's aligned text wins.
            if cid not in meta:
                meta[cid] = {
                    'doc':    docs[rank]   if rank < len(docs)   else '',
                    'source': srcs[rank]   if rank < len(srcs)   else '',
                    'text':   texts[rank]  if rank < len(texts)  else '',
                    'title':  titles[rank] if rank < len(titles) else '',
                }
    top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
    return {
        'retrieved_ids':     top_ids,
        'retrieved_docs':    [meta[cid]['doc']    for cid in top_ids],
        'retrieved_sources': [meta[cid]['source'] for cid in top_ids],
        'retrieved_texts':   [meta[cid]['text']   for cid in top_ids],
        'retrieved_titles':  [meta[cid]['title']  for cid in top_ids],
        'retrieved_scores':  [scores[cid]         for cid in top_ids],
        'score_name': 'rrf',
    }

## Build indices

In [ ]:
#@title 4.1 Build original BM25 index
print('Building BM25 on original corpus text...')
bm25_text = build_bm25(c_text, tok, desc='BM25-original')
print('BM25 ready.')

In [ ]:
#@title 4.2 Dense retriever selection — sequential OOM-safe version

import gc, time, torch
from sentence_transformers import SentenceTransformer
import faiss

# Active models. For low-memory runs, temporarily leave only BGE-M3.
DENSE_MODELS = {
    'BGE-M3': 'BAAI/bge-m3',
    'E5-Large': 'intfloat/multilingual-e5-large',
    'LaBSE': 'sentence-transformers/LaBSE',
}

DENSE_BATCH_SIZE = 128      # 96 GB VRAM easily handles this at seq_len 512
MAX_SEQ_LENGTH = 512        # covers your longest chunks (~450 words), no truncation


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def model_prefixes(model_id):
    m = model_id.lower()
    if 'e5' in m:
        return 'passage: ', 'query: '
    return '', ''


def encode_texts_safe(model, texts, prefix='', batch_size=DENSE_BATCH_SIZE):
    prepared = [prefix + str(t) for t in texts]
    with torch.inference_mode():
        emb = model.encode(
            prepared,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=DEVICE,
        )
    return emb.astype('float32')


def build_faiss(embeddings):
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    return index


def evaluate_dense_model(name, model_id):
    """Load one model, evaluate it, then release GPU memory.
    This avoids keeping three large transformer models in memory at the same time.
    """
    print(f'\nLoading dense model: {name} — {model_id}')
    clear_gpu()

    model = SentenceTransformer(model_id, device=DEVICE, trust_remote_code=True)
    model.max_seq_length = MAX_SEQ_LENGTH

    corpus_prefix, query_prefix = model_prefixes(model_id)

    print('Encoding corpus...')
    corpus_emb = encode_texts_safe(model, c_text, corpus_prefix)
    faiss_index_tmp = build_faiss(corpus_emb)

    print('Encoding answerable questions...')
    q_emb_ans_tmp = encode_texts_safe(model, bench_ans['question'].tolist(), query_prefix)

    t0 = time.time()
    res = [dense_ret(faiss_index_tmp, q_emb_ans_tmp[i], k=10) for i in range(len(bench_ans))]
    latency = (time.time() - t0) / max(1, len(bench_ans)) * 1000

    metrics, details = eval_retrieval(res, bench_ans)
    metrics.update({
        'Model': name,
        'ModelId': model_id,
        'Latency_ms': round(latency, 2),
        'MaxSeqLength': MAX_SEQ_LENGTH,
        'BatchSize': DENSE_BATCH_SIZE,
    })

    print({k: metrics[k] for k in ['Hit@1', 'Recall@10', 'MRR@10', 'nDCG@10']})

    # Save only details and metrics; do not keep model/faiss_index for non-best models.
    save_table(details.assign(Model=name), f'detail01_dense_{name.replace("/", "_").replace(" ", "_")}')

    del model, corpus_emb, faiss_index_tmp, q_emb_ans_tmp, res
    clear_gpu()

    return metrics


dense_rows = []

for name, model_id in DENSE_MODELS.items():
    try:
        dense_rows.append(evaluate_dense_model(name, model_id))
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'CUDA OOM for {name}. Skipping this model. Try Runtime → Restart session or reduce MAX_SEQ_LENGTH.')
            clear_gpu()
        else:
            raise


dense_df = summarize_metric_table(dense_rows, 'table01_dense_retriever_selection')

if len(dense_df) == 0:
    raise RuntimeError('No dense model completed successfully. Use CPU runtime or keep only one model in DENSE_MODELS.')

BEST_DENSE = dense_df.sort_values(['Recall@10', 'MRR@10', 'nDCG@10'], ascending=False).iloc[0]['Model']
best_model_id = dense_df.loc[dense_df['Model'] == BEST_DENSE, 'ModelId'].iloc[0]
best_corpus_prefix, best_query_prefix = model_prefixes(best_model_id)

print(f'Best dense retriever by Recall@10/MRR/nDCG: {BEST_DENSE} — {best_model_id}')

# Reload only the best model for downstream experiments.
clear_gpu()
best_model = SentenceTransformer(best_model_id, device=DEVICE, trust_remote_code=True)
best_model.max_seq_length = MAX_SEQ_LENGTH

print('Re-encoding corpus with the best dense model for downstream experiments...')
best_corpus_emb = encode_texts_safe(best_model, c_text, best_corpus_prefix)
faiss_index = build_faiss(best_corpus_emb)

print('Encoding answerable questions with the best dense model...')
query_embs_ans = encode_texts_safe(best_model, bench_ans['question'].tolist(), best_query_prefix)

print('Encoding all 242 benchmark questions with the best dense model...')
query_embs_all = encode_texts_safe(best_model, bench_all['question'].tolist(), best_query_prefix)

del best_corpus_emb
clear_gpu()

print('Done.')


In [ ]:
#@title 4.2b Add Qwen3 to the dense retriever comparison
# These two were missing because each needs special loading:
#   - Qwen3-Embedding-0.6B: requires a query instruction prefix for good retrieval
import gc, time
import numpy as np, pandas as pd
import torch
from sentence_transformers import SentenceTransformer

EXTRA_DENSE = {
    'Qwen3-0.6B': 'Qwen/Qwen3-Embedding-0.6B',
}

QWEN3_QUERY_INSTRUCTION = (
    'Instruct: Given a Kazakh legal question, retrieve relevant legal provisions\nQuery: '
)

def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def encode_corpus_and_eval(name, model_id):
    print(f'\n=== {name} ({model_id}) ===')
    is_jina = 'jina' in model_id.lower()
    is_qwen = 'qwen' in model_id.lower()

    st_kwargs = dict(device=DEVICE)
    if is_jina:
        st_kwargs['trust_remote_code'] = True
    model = SentenceTransformer(model_id, **st_kwargs)
    try:
        model.max_seq_length = MAX_SEQ_LENGTH
    except Exception:
        pass

    enc_kwargs = dict(batch_size=DENSE_BATCH_SIZE, show_progress_bar=True,
                      convert_to_numpy=True, normalize_embeddings=True)

    # corpus
    if is_jina:
        try:
            corpus_emb = model.encode(c_text, prompt_name='retrieval.passage', **enc_kwargs)
        except Exception:
            corpus_emb = model.encode(c_text, **enc_kwargs)
    else:
        corpus_emb = model.encode(c_text, **enc_kwargs)
    corpus_emb = corpus_emb.astype('float32')

    # queries
    questions = bench_ans['question'].tolist()
    if is_qwen:
        query_emb = model.encode([QWEN3_QUERY_INSTRUCTION + q for q in questions], **enc_kwargs)
    elif is_jina:
        try:
            query_emb = model.encode(questions, prompt_name='retrieval.query', **enc_kwargs)
        except Exception:
            query_emb = model.encode(questions, **enc_kwargs)
    else:
        query_emb = model.encode(questions, **enc_kwargs)
    query_emb = query_emb.astype('float32')

    import faiss
    index = faiss.IndexFlatIP(corpus_emb.shape[1])
    index.add(corpus_emb)

    start = time.time()
    results = [dense_ret(index, query_emb[i], k=10) for i in range(len(bench_ans))]
    lat = (time.time() - start) / len(bench_ans) * 1000

    metrics, details = eval_retrieval(results, bench_ans)
    metrics.update({'Model': name, 'ModelId': model_id, 'Latency_ms': round(lat, 2),
                    'MaxSeqLength': MAX_SEQ_LENGTH, 'BatchSize': DENSE_BATCH_SIZE})
    save_table(details, f'detail01_dense_{name}')
    print({k: metrics[k] for k in ['Hit@1','Recall@10','MRR@10','nDCG@10']})

    del model, corpus_emb, query_emb, index
    _free()
    return metrics

extra_rows = []
for name, mid in EXTRA_DENSE.items():
    try:
        extra_rows.append(encode_corpus_and_eval(name, mid))
    except Exception as e:
        print(f'FAILED {name}: {type(e).__name__}: {e}')

if extra_rows:
    new_df = pd.DataFrame(extra_rows)
    try:
        base = pd.read_csv(f'{OUTPUT_DIR}/table01_dense_retriever_selection.csv')
    except Exception:
        base = pd.DataFrame()
    combined = pd.concat([base, new_df], ignore_index=True)
    front = ['Model','Hit@1','Recall@1','Recall@3','Recall@5','Recall@10','MRR@10','nDCG@10','Latency_ms']
    cols = [c for c in front if c in combined.columns] + [c for c in combined.columns if c not in front]
    combined = combined[cols].sort_values('Recall@10', ascending=False).reset_index(drop=True)
    display(combined)
    save_table(combined, 'table01_dense_retriever_selection_full')
    print('Merged. Licenses for paper: Jina-v3 = CC-BY-NC-4.0 (non-commercial), Qwen3 = Apache-2.0.')
else:
    print('No extra models succeeded.')

## Table 16 — Dense backbone comparison (BGE-M3 selected)

In [ ]:
#@title 4.3 BM25, Dense, and Hybrid comparison on 182 answerable questions
retrieval_rows = []
retrieval_details = {}

# Naive BM25 k=5
start = time.time()
bm25_k5 = [bm25_ret(bm25_text, row['question'], k=5, tokenizer=tok) for _, row in tqdm(bench_ans.iterrows(), total=len(bench_ans), desc='BM25 k=5')]
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(bm25_k5, bench_ans)
metrics.update({'System': 'Naive RAG: BM25 k=5', 'RetrievalType': 'lexical', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details['Naive RAG: BM25 k=5'] = details

# BM25 k=10
start = time.time()
bm25_k10 = [bm25_ret(bm25_text, row['question'], k=10, tokenizer=tok) for _, row in tqdm(bench_ans.iterrows(), total=len(bench_ans), desc='BM25 k=10')]
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(bm25_k10, bench_ans)
metrics.update({'System': 'BM25 k=10', 'RetrievalType': 'lexical', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details['BM25 k=10'] = details

# Dense best
start = time.time()
dense_k10 = [dense_ret(faiss_index, query_embs_ans[i], k=10) for i in tqdm(range(len(bench_ans)), desc=f'Dense {BEST_DENSE}')]
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(dense_k10, bench_ans)
metrics.update({'System': f'Dense {BEST_DENSE} k=10', 'RetrievalType': 'semantic', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details[f'Dense {BEST_DENSE} k=10'] = details

# Static Hybrid BM25 + Dense through RRF, k=10
start = time.time()
hybrid_k10 = []
for i, row in tqdm(list(bench_ans.iterrows()), desc='Hybrid k=10'):
    q = row['question']
    hybrid_k10.append(rrf([bm25_ret(bm25_text, q, k=10), dense_ret(faiss_index, query_embs_ans[i], k=10)], k=10))
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(hybrid_k10, bench_ans)
metrics.update({'System': 'Static Hybrid RAG: BM25+BGE-M3 RRF k=10', 'RetrievalType': 'hybrid', 'Latency_ms': round(lat, 2)})
retrieval_rows.append(metrics)
retrieval_details['Static Hybrid RAG'] = details

retrieval_df = summarize_metric_table(retrieval_rows, 'table02_retrieval_baselines')
save_table(pd.concat([d.assign(System=s) for s, d in retrieval_details.items()], ignore_index=True), 'detail02_retrieval_baselines')

# --- Save this run's retrieval summary tagged by ITERATION for cross-run reproducibility ---
ITERATION = int(os.environ.get('LEGALRAG_ITERATION', '1'))  # set 2 on the second run
retrieval_df.assign(iteration=ITERATION).to_csv(
    f'/content/retrieval_iter{ITERATION}.csv', index=False, encoding='utf-8-sig')
print(f'Saved /content/retrieval_iter{ITERATION}.csv  (set LEGALRAG_ITERATION=2 before the 2nd run)')


## Table 18 — Kazakh character folding (BM25)

In [ ]:
#@title 5.1 Build normalized/folded BM25 index — safe version

KZ_FOLD = {
    'ә': 'а', 'і': 'и', 'ң': 'н', 'ғ': 'г',
    'ү': 'у', 'ұ': 'у', 'қ': 'к', 'ө': 'о', 'һ': 'х'
}


def fold_kazakh(text):
    s = str(text).lower()
    for a, b in KZ_FOLD.items():
        s = s.replace(a, b)
    return s


def tok_fold(text):
    return fold_kazakh(text).split()


def is_usable_text_column(texts, tokenizer, min_nonempty_ratio=0.05):
    nonempty = 0
    total = len(texts)

    for t in texts:
        try:
            if len(tokenizer(str(t))) > 0:
                nonempty += 1
        except Exception:
            pass

    ratio = nonempty / max(1, total)
    print(f'Non-empty tokenized docs: {nonempty}/{total} = {ratio:.4f}')
    return ratio >= min_nonempty_ratio


# 1) Try corpus normalized_text only if it is actually usable
use_corpus_normalized = False

if NORM_TEXT_COL and NORM_TEXT_COL in corpus_df.columns:
    candidate_texts = corpus_df[NORM_TEXT_COL].fillna('').astype(str).tolist()
    print(f'Found corpus normalized field: {NORM_TEXT_COL}')

    if is_usable_text_column(candidate_texts, tok):
        use_corpus_normalized = True
    else:
        print(f'Column {NORM_TEXT_COL} exists but is not usable for BM25. Falling back to Kazakh character folding.')
else:
    print('No corpus normalized_text column found. Falling back to Kazakh character folding.')


# 2) Select normalized/folded representation
if use_corpus_normalized:
    normalized_texts = candidate_texts
    normalized_label = 'BM25-normalized_text'
    normalized_tokenizer = tok

    def normalize_query_for_bm25(q):
        return str(q)

else:
    normalized_texts = [fold_kazakh(t) for t in c_text]
    normalized_label = 'BM25-fold Kazakh characters'
    normalized_tokenizer = tok

    def normalize_query_for_bm25(q):
        return fold_kazakh(q)


# 3) Final safety check before BM25
if not is_usable_text_column(normalized_texts, normalized_tokenizer, min_nonempty_ratio=0.05):
    raise ValueError(
        'Normalized/folded texts are still empty after fallback. '
        'Check c_text and corpus text column.'
    )


# 4) Build BM25
bm25_norm = build_bm25(
    normalized_texts,
    normalized_tokenizer,
    desc=normalized_label
)


# 5) Evaluate normalized/folded BM25
start = time.time()
bm25_norm_k10 = []

for _, row in tqdm(bench_ans.iterrows(), total=len(bench_ans), desc=normalized_label):
    q = normalize_query_for_bm25(row['question'])
    bm25_norm_k10.append(
        bm25_ret(
            bm25_norm,
            q,
            k=10,
            tokenizer=normalized_tokenizer
        )
    )

lat = (time.time() - start) / max(1, len(bench_ans)) * 1000


# 6) Compare original vs normalized/folded
m_orig, d_orig = eval_retrieval(bm25_k10, bench_ans)
m_norm, d_norm = eval_retrieval(bm25_norm_k10, bench_ans)

norm_rows = []

for system, metrics, latency in [
    ('BM25-original text', m_orig, np.nan),
    (normalized_label, m_norm, round(lat, 2)),
]:
    row = {
        'System': system,
        'Latency_ms': latency
    }

    for k in [
        'Hit@1', 'Recall@5', 'Recall@10',
        'MRR@10', 'nDCG@10',
        'DocRecall@10', 'SourceRecall@10'
    ]:
        if k in metrics:
            row[k] = metrics[k]

    norm_rows.append(row)

norm_df = pd.DataFrame(norm_rows)

metric_cols = [
    'Hit@1', 'Recall@5', 'Recall@10',
    'MRR@10', 'nDCG@10',
    'DocRecall@10', 'SourceRecall@10'
]

delta = {
    'System': f'Delta: {normalized_label} minus original',
    'Latency_ms': np.nan
}

for c in metric_cols:
    if c in norm_df.columns:
        delta[c] = round(float(norm_df.iloc[1][c] - norm_df.iloc[0][c]), 4)

norm_df = pd.concat([norm_df, pd.DataFrame([delta])], ignore_index=True)

display(norm_df)

save_table(norm_df, 'table03_text_normalization_or_folding')
save_table(
    d_norm.assign(System=normalized_label),
    'detail03_text_normalization_or_folding'
)

## Table 19 — Second-stage reranking ablation (BGE-reranker-v2-m3)

In [ ]:
#@title 6.1 Reranking ablation — FIXED (use texts/docs from RRF result, not id2* lookup)
from sentence_transformers import CrossEncoder

RUN_RERANKING = True
RERANKER_ID = 'BAAI/bge-reranker-v2-m3'
RERANK_POOLS = [20, 50, 100]

rerank_rows = []
rerank_details = {}

# Fair no-rerank baseline.
start = time.time()
hybrid_no_rerank = []
for i, row in tqdm(list(bench_ans.iterrows()), desc='Hybrid no rerank'):
    q = row['question']
    hybrid_no_rerank.append(rrf([bm25_ret(bm25_text, q, 10), dense_ret(faiss_index, query_embs_ans[i], 10)], k=10))
lat = (time.time() - start) / len(bench_ans) * 1000
metrics, details = eval_retrieval(hybrid_no_rerank, bench_ans)
rerank_rows.append({'Pipeline': 'Hybrid only top-10', 'CandidatePool': 10, 'FinalTopK': 10, 'Latency_ms': round(lat, 2), **metrics})
rerank_details['Hybrid only top-10'] = details

if RUN_RERANKING:
    print(f'Loading reranker: {RERANKER_ID}')
    reranker = CrossEncoder(RERANKER_ID, device=DEVICE, max_length=512)

    def rerank_result(question, res, final_k=10):
        """Reorder candidates with a cross-encoder.
        FIX: take texts/docs/sources DIRECTLY from the RRF result (already aligned
        with retrieved_ids), instead of re-looking-up id2text/id2doc which can
        mismatch on key type or return truncated text. This keeps ids<->text<->doc
        perfectly synchronized, so reranked ids and their docs stay correct."""
        cids   = res.get('retrieved_ids', [])
        texts  = res.get('retrieved_texts', [])
        docs   = res.get('retrieved_docs', [])
        srcs   = res.get('retrieved_sources', [])
        titles = res.get('retrieved_titles', [])
        if not cids:
            return res
        # guard: align lengths (fall back to id2text only if RRF text missing)
        n = len(cids)
        def at(lst, i, fallback=''):
            return lst[i] if i < len(lst) and lst[i] not in (None, '') else fallback
        pair_texts = [at(texts, i) or id2text.get(cids[i], '') for i in range(n)]
        pairs = [(question, t) for t in pair_texts]
        scores = np.asarray(reranker.predict(pairs), dtype=float)
        order = np.argsort(scores)[::-1][:final_k]
        return {
            'retrieved_ids':     [cids[i] for i in order],
            'retrieved_docs':    [at(docs, i,   id2doc.get(cids[i], ''))   for i in order],
            'retrieved_sources': [at(srcs, i,   id2src.get(cids[i], ''))   for i in order],
            'retrieved_texts':   [pair_texts[i]                            for i in order],
            'retrieved_titles':  [at(titles, i, id2title.get(cids[i], '')) for i in order],
            'retrieved_scores':  [float(scores[i]) for i in order],
            'score_name': 'cross_encoder',
        }

    for pool in RERANK_POOLS:
        start = time.time()
        reranked = []
        for i, row in tqdm(list(bench_ans.iterrows()), desc=f'Hybrid -> reranker top-{pool}'):
            q = row['question']
            cands = rrf([bm25_ret(bm25_text, q, pool), dense_ret(faiss_index, query_embs_ans[i], pool)], k=pool)
            reranked.append(rerank_result(q, cands, final_k=10))
        lat = (time.time() - start) / len(bench_ans) * 1000
        metrics, details = eval_retrieval(reranked, bench_ans)
        rerank_rows.append({'Pipeline': f'Hybrid -> reranker top-{pool}', 'CandidatePool': pool, 'FinalTopK': 10, 'Latency_ms': round(lat, 2), **metrics})
        rerank_details[f'Hybrid -> reranker top-{pool}'] = details

rerank_df = pd.DataFrame(rerank_rows)
display(rerank_df)
save_table(rerank_df, 'table04_reranking_ablation')
save_table(pd.concat([d.assign(Pipeline=s) for s, d in rerank_details.items()], ignore_index=True), 'detail04_reranking_ablation')

if len(rerank_df) > 1:
    baseline_hit = float(rerank_df.iloc[0]['Hit@1'])
    best_rerank_hit = float(rerank_df.iloc[1:]['Hit@1'].max())
    print('Reranker improves Hit@1:', best_rerank_hit > baseline_hit,
          f'(baseline {baseline_hit:.3f} vs best rerank {best_rerank_hit:.3f})')
